In [1]:
import pandas as pd
import numpy as np
import os
from db_connection import SQLConnection
from define_environment import load_correct_environment_variables
from collections.abc import Iterable

In [2]:
load_correct_environment_variables()

In [3]:
db = SQLConnection(os.environ.get("POSTGRES_USER"), os.environ.get("POSTGRES_PASSWORD"), os.environ.get("POSTGRES_CONTAINER"), os.environ.get("POSTGRES_PORT"), os.environ.get("POSTGRES_DB"))

In [4]:
# Glicko-2 implementation for multiple stats (e.g., goals, assists) in a football match.
# Sample DataFrame with match data and player stats
# Replace with your actual data
data = pd.DataFrame({
    'match_id': [1]*6 + [2]*6,
    'team': ['TeamA']*3 + ['TeamB']*3 + ['TeamA']*3 + ['TeamB']*3,
    'player': ['Player1', 'Player2', 'Player3', 'Player4', 'Player5', 'Player6']*2,
    'position': ['forward', 'midfield', 'defender', 'defender', 'midfield', 'forward']*2,
    'side': ['left', 'central', 'right', 'right', 'central', 'left']*2,
    'goals': [1, 0, 0, 0, 1, 0, 0, 1, 0, 2, 0, 0],
    'assists': [0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1],
    # Add other stats columns as needed
})

# List of stats columns to process
stats_columns = [
    "goals",
    "assists",
    # Add other stats columns as needed
]

# Initialize player ratings, RD, and volatility
players = data['player'].unique()
initial_rating = 1500  # Initial rating
initial_RD = 350       # Initial rating deviation (high uncertainty)
initial_volatility = 0.06  # Initial volatility
tau = 0.5  # Volatility parameter (controls fluctuation)

# Create a nested dictionary for ratings
ratings = {stat: {player: {'rating': initial_rating,
                           'RD': initial_RD,
                           'vol': initial_volatility} for player in players} for stat in stats_columns}

# Glicko-2 constants
q = np.log(10) / 400  # q = ln(10)/400 ≈ 0.0057565

def g(phi):
    """Calculates the g(phi) factor."""
    return 1 / np.sqrt(1 + (3 * phi ** 2) / (np.pi ** 2))

def E(mu, mu_j, phi_j):
    """Calculates the expected score."""
    return 1 / (1 + np.exp(-g(phi_j) * (mu - mu_j)))

def f(x, delta, phi, v, sigma, tau):
    """Function to solve for new volatility (sigma)."""
    a = np.log(sigma ** 2)
    ex = np.exp(x)
    numerator = ex * (delta ** 2 - phi ** 2 - v - ex)
    denominator = 2 * (phi ** 2 + v + ex) ** 2
    return (numerator / denominator) - ((x - a) / (tau ** 2))

def position_weight(player_position, player_side, opponent_position, opponent_side):
    """
    Calculate weight based on positional interactions.

    Returns a weight between 0 and 1.
    """
    # Position interactions
    position_interactions = {
        'forward': {'defender': 1.0, 'midfield': 0.5, 'forward': 0.1},
        'midfield': {'defender': 0.5, 'midfield': 1.0, 'forward': 0.5},
        'defender': {'defender': 0.1, 'midfield': 0.5, 'forward': 1.0},
    }

    # Side interactions
    side_interactions = {
        'left': {'left': 1.0, 'central': 0.5, 'right': 0.1},
        'central': {'left': 0.5, 'central': 1.0, 'right': 0.5},
        'right': {'left': 0.1, 'central': 0.5, 'right': 1.0},
    }

    # Get position weight
    pos_weight = position_interactions.get(player_position, {}).get(opponent_position, 0.5)

    # Get side weight
    side_weight = side_interactions.get(player_side, {}).get(opponent_side, 0.5)

    # Combine weights (average of position and side weights)
    weight = (pos_weight + side_weight) / 2

    return weight

# Process matches in chronological order
match_ids = sorted(data['match_id'].unique())

for match_id in match_ids:
    # Extract data for the current match
    match_data = data[data['match_id'] == match_id]
    teams = match_data['team'].unique()
    if len(teams) != 2:
        continue  # Skip if not exactly two teams

    # Identify teams and their players
    team1, team2 = teams
    team1_players = match_data[match_data['team'] == team1]
    team2_players = match_data[match_data['team'] == team2]

    # Process each player in the match
    for idx, row in match_data.iterrows():
        player = row['player']
        team = row['team']
        player_position = row['position']
        player_side = row['side']
        opponent_team = team2_players if team == team1 else team1_players

        for stat in stats_columns:
            # Skip if the stat value is NaN
            if pd.isna(row[stat]):
                continue

            # Retrieve player's current rating, RD, and volatility
            rating_dict = ratings[stat][player]
            rating_i = rating_dict['rating']
            RD_i = rating_dict['RD']
            sigma_i = rating_dict['vol']

            # Convert to Glicko-2 scale
            mu_i = (rating_i - 1500) / 173.7178
            phi_i = RD_i / 173.7178
            sigma_i = sigma_i  # Volatility remains unchanged

            # Initialize sums for variance and delta calculations
            sum_g_E = 0
            sum_g_s_E = 0

            # Iterate over opponents
            for opp_idx, opp_row in opponent_team.iterrows():
                opponent = opp_row['player']
                opponent_position = opp_row['position']
                opponent_side = opp_row['side']
                opponent_rating = ratings[stat][opponent]['rating']
                opponent_RD = ratings[stat][opponent]['RD']
                mu_j = (opponent_rating - 1500) / 173.7178
                phi_j = opponent_RD / 173.7178

                # Calculate weight based on positional interactions
                weight = position_weight(player_position, player_side, opponent_position, opponent_side)

                # Compute g(phi_j) and E(mu_i, mu_j, phi_j)
                g_phi_j = g(phi_j)
                E_ij = E(mu_i, mu_j, phi_j)

                # Actual performance: Normalize the stat value between 0 and 1
                max_stat_value = data[stat].max()
                if max_stat_value == 0:
                    continue  # Avoid division by zero
                actual_performance = row[stat] / max_stat_value

                # Update sums with weights
                sum_g_E += weight * (g_phi_j ** 2) * E_ij * (1 - E_ij)
                sum_g_s_E += weight * g_phi_j * (actual_performance - E_ij)

            if sum_g_E == 0:
                continue  # Avoid division by zero

            # Compute variance v
            v = 1 / sum_g_E

            # Compute delta (Δ)
            delta = v * sum_g_s_E

            # Update volatility (σ') using iterative method
            a = np.log(sigma_i ** 2)
            A = a
            B = None
            if delta ** 2 > phi_i ** 2 + v:
                B = np.log(delta ** 2 - phi_i ** 2 - v)
            else:
                k = 1
                while f(a - k * tau, delta, phi_i, v, sigma_i, tau) < 0:
                    k += 1
                B = a - k * tau

            # Newton-Raphson iteration to find σ'
            epsilon = 1e-6
            fA = f(A, delta, phi_i, v, sigma_i, tau)
            fB = f(B, delta, phi_i, v, sigma_i, tau)
            while abs(B - A) > epsilon:
                C = A + (A - B) * fA / (fB - fA)
                fC = f(C, delta, phi_i, v, sigma_i, tau)
                if fC * fB < 0:
                    A = B
                    fA = fB
                else:
                    fA /= 2
                B = C
                fB = fC

            sigma_prime = np.exp(A / 2)

            # Update phi* (intermediate RD)
            phi_star = np.sqrt(phi_i ** 2 + sigma_prime ** 2)

            # Update phi' (new RD)
            phi_prime = 1 / np.sqrt( (1 / phi_star ** 2) + (1 / v) )

            # Update mu' (new rating)
            mu_prime = mu_i + phi_prime ** 2 * sum_g_s_E

            # Convert back to original scale
            rating_prime = mu_prime * 173.7178 + 1500
            RD_prime = phi_prime * 173.7178

            # Update player's rating, RD, and volatility
            ratings[stat][player]['rating'] = rating_prime
            ratings[stat][player]['RD'] = RD_prime
            ratings[stat][player]['vol'] = sigma_prime

# Display the final ratings
for stat in stats_columns:
    print(f"\nFinal ratings for stat '{stat}':")
    for player, info in ratings[stat].items():
        print(f"  {player}: Rating={info['rating']:.2f}, RD={info['RD']:.2f}, Volatility={info['vol']:.5f}")


Final ratings for stat 'goals':
  Player1: Rating=1234.72, RD=223.84, Volatility=0.06000
  Player2: Rating=1268.87, RD=200.17, Volatility=0.06000
  Player3: Rating=1106.49, RD=214.97, Volatility=0.06000
  Player4: Rating=1353.87, RD=205.77, Volatility=0.06000
  Player5: Rating=1140.25, RD=197.10, Volatility=0.06000
  Player6: Rating=1030.97, RD=205.77, Volatility=0.06000

Final ratings for stat 'assists':
  Player1: Rating=1484.25, RD=221.46, Volatility=0.06000
  Player2: Rating=1388.88, RD=221.59, Volatility=0.06001
  Player3: Rating=1136.15, RD=221.46, Volatility=0.06000
  Player4: Rating=1379.39, RD=229.85, Volatility=0.06001
  Player5: Rating=1089.99, RD=199.29, Volatility=0.06000
  Player6: Rating=1433.33, RD=214.85, Volatility=0.06000


In [ ]:
# Get match log data
game_data = db.get_df("SELECT * FROM match_logs ORDER BY date, match_id")
game_data

In [6]:
# get player data
player_data = db.get_df("SELECT * FROM player")
player_data

,id,first_name,last_name,birth_year,position,nationality,fbref_match_logs_href,fbref_id
0,p-00001,Ricky,Holmes,1987.0,MF,eng ENG,https://fbref.com/en/players/27650ab0/matchlog...,27650ab0
1,p-00002,Freddie,Hinds,1999.0,FW,eng ENG,https://fbref.com/en/players/01e2def7/matchlog...,01e2def7
2,p-00003,Abel,Hernández,1990.0,FW,uy URU,https://fbref.com/en/players/a36da810/matchlog...,a36da810
3,p-00004,Mark,Howard,1986.0,GK,eng ENG,https://fbref.com/en/players/d5d89b53/matchlog...,d5d89b53
4,p-00005,Lloyd,Isgrove,1993.0,"MF,FW",wls WAL,https://fbref.com/en/players/430696d5/matchlog...,430696d5
...,...,...,...,...,...,...,...,...
3168,p-03080,Lucas,Paquetá,1997.0,MF,br BRA,https://fbref.com/en/players/9b6f7fd5/matchlog...,9b6f7fd5
3169,p-02966,Marcus,Tavernier,1999.0,"MF,FW",eng ENG,https://fbref.com/en/players/5c0da4a4/matchlog...,5c0da4a4
3170,p-02085,Boubacar,Traoré,2001.0,MF,ml MLI,https://fbref.com/en/players/999ceb5f/matchlog...,999ceb5f
3171,p-02876,Nicolás,Domínguez,1998.0,"MF,FW",ar ARG,https://fbref.com/en/players/20b3a502/matchlog...,20b3a502


In [7]:
# unique positions
game_data['position'].unique()

array(['AM,FW', 'DM,CM', None, 'CM', 'CB', 'WB,LB', 'RW', 'CM,DM', 'LM',
       'FW', 'RB', 'LB', 'GK', 'FW,LW,LM', 'WB,RB', 'RM', 'AM,RM', 'DM',
       'LW', 'LW,RW', 'RM,LM', 'LM,LW', 'WB,FW,LW', 'FW,LW', 'WB', 'AM',
       'RB,CB', 'CM,RM', 'AM,RW', 'AM,CM', 'CM,LM', 'LW,FW', 'FW,RM',
       'LM,RM', 'CM,AM', 'RW,FW', 'LW,AM', 'FW,RW', 'LW,LM', 'RW,RM',
       'LW,LM,AM', 'AM,CM,FW', 'DM,AM', 'LW,RW,AM', 'LB,CB', 'RW,RM,WB',
       'RW,AM', 'RB,WB', 'RW,RB', 'RW,CM', 'LW,LB', 'FW,CB', 'AM,LW',
       'AM,DM', 'RB,RM', 'CM,LW', 'LM,FW', 'RM,FW', 'FW,LM', 'FW,AM',
       'RW,LW,LM', 'LM,DM,CM', 'RW,LM', 'RW,LW', 'LM,CM', 'CB,WB',
       'CM,CB', 'CM,LB', 'DM,LB', 'LB,WB', 'RM,CM', 'RB,CM', 'LB,RB',
       'LW,AM,LM', 'AM,DM,CM', 'AM,RM,LW', 'DM,LW', 'CB,RB', 'RW,WB,CM',
       'CM,FW', 'LM,RW,RM', 'RM,RW', 'LW,CM', 'CB,LM', 'LM,RW', 'WB,RW',
       'RW,CB', 'RB,WB,DM', 'LB,WB,DM', 'CM,DM,FW', 'LM,FW,WB',
       'RM,WB,DM', 'DM,RM', 'LM,DM', 'CB,LM,CM', 'CM,AM,RM', 'RM,DM,LM',
       '

In [8]:
# Check for missing positions
print("BEFORE - Entries with no position: " + str(len(game_data[game_data['position'].isnull()][["player_id", "position"]])/len(game_data)))

# Merge game_data with player_data to get the position from player_data
merged_data = game_data.merge(player_data[['id', 'position']], left_on='player_id', right_on='id', how='left', suffixes=('', '_player'))

# Replace None values in game_data['position'] with the corresponding values from player_data['position']
game_data['position'] = np.where(game_data['position'].isnull(), merged_data['position_player'], game_data['position'])

print("AFTER - Entries with no position: " + str(len(game_data[game_data['position'].isnull()][["player_id", "position"]])/len(game_data)))
print("Entries with multiple positions: " + str(len(game_data[game_data['position'].str.contains(',')][["player_id", "position"]])/len(game_data)))
players_with_multiple_positions = game_data[game_data['position'].str.contains(',')][["player_id", "position"]]

BEFORE - Entries with no position: 0.2254871481150903
AFTER - Entries with no position: 0.0
Entries with multiple positions: 0.16138316969703048


In [9]:
# For players with multiple positions, keep the most frequent one and replace the entries with multiple positions
game_data['position'] = game_data['position'].apply(lambda x: x.split(',') if ',' in x else x)
for id in players_with_multiple_positions['player_id'].unique():
	if not any(isinstance(pos, Iterable) and not isinstance(pos, (str, bytes)) for pos in game_data['position']):
		continue
	player: pd.DataFrame = game_data[game_data['player_id'] == id].copy()
	flattened_positions = []
	for idx, row in player.iterrows():
		if isinstance(row["position"], Iterable) and not isinstance(row["position"], (str, bytes)):
			flattened_positions.extend(row["position"])
			most_frequent_position: str = max(set(flattened_positions), key=flattened_positions.count)
			print(f"Player {id} has multiple positions: {row['position']} at index {idx} - Most frequent: {most_frequent_position}")
			game_data.loc[idx, "position"] = most_frequent_position
		else:
			flattened_positions.append(row["position"])

Player p-02941 has multiple positions: ['AM', 'FW'] at index 0 - Most frequent: FW
Player p-02941 has multiple positions: ['AM', 'LW'] at index 404 - Most frequent: AM
Player p-02941 has multiple positions: ['FW', 'RW'] at index 926 - Most frequent: FW
Player p-02941 has multiple positions: ['WB', 'RW'] at index 4967 - Most frequent: AM
Player p-02941 has multiple positions: ['RM', 'RW'] at index 6077 - Most frequent: AM
Player p-02941 has multiple positions: ['RW', 'LW'] at index 6292 - Most frequent: AM
Player p-02941 has multiple positions: ['FW', 'AM'] at index 7637 - Most frequent: AM
Player p-02941 has multiple positions: ['AM', 'LW'] at index 7938 - Most frequent: AM
Player p-02941 has multiple positions: ['RW', 'AM'] at index 11578 - Most frequent: AM
Player p-02941 has multiple positions: ['LW', 'RW'] at index 11844 - Most frequent: AM
Player p-02941 has multiple positions: ['LW', 'RW'] at index 15388 - Most frequent: LW
Player p-02941 has multiple positions: ['LW', 'AM'] at i

In [10]:
print("\nEntries with multiple positions: " + str(len(game_data[game_data['position'].apply(
        lambda x: isinstance(x, Iterable) and not isinstance(x, (str, bytes))
    )][["player_id", "position"]])/len(game_data)))


Entries with multiple positions: 0.0


In [11]:
game_data['position'].value_counts()

position
FW    16473
CB    13374
CM     9257
GK     8741
DF     8635
LB     5455
RB     5404
RW     4674
MF     4626
DM     4473
LW     4466
AM     4354
LM     4087
RM     3728
WB     1966
Name: count, dtype: int64

In [12]:
# Only use data from the 2024-2025 season as the sample data
season_data = game_data[game_data['season'] == '2024-2025']
season_data

,player_id,fbref_id,competition_id,match_id,season,date,location,team_id,opponent_id,position,...,gk_pens_allowed,gk_pens_saved,gk_pens_missed,gk_passed_completed_launched,gk_passes_launched,gk_passes,gk_passes_throws,gk_passes_length_avg,gk_goal_kicks,gk_goal_kicks_length_avg
97175,p-02911,4f9091bc,x-00005,m-02661,2024-2025,2024-08-16,Home,t-00036,t-00026,MF,...,0,0,0,0,0,0,0,0.0,0,0.0
97176,p-02853,99127249,x-00005,m-02661,2024-2025,2024-08-16,Home,t-00036,t-00026,FW,...,0,0,0,0,0,0,0,0.0,0,0.0
97177,p-02806,a36524bf,x-00005,m-02661,2024-2025,2024-08-16,Away,t-00026,t-00036,CB,...,0,0,0,0,0,0,0,0.0,0,0.0
97178,p-02829,a8748947,x-00005,m-02661,2024-2025,2024-08-16,Away,t-00026,t-00036,AM,...,0,0,0,0,0,0,0,0.0,0,0.0
97179,p-02897,4d224fe8,x-00005,m-02661,2024-2025,2024-08-16,Home,t-00036,t-00026,DM,...,0,0,0,0,0,0,0,0.0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99708,p-02802,d38fdf53,x-00005,m-02730,2024-2025,2024-10-06,Home,t-00012,t-00064,LB,...,0,0,0,0,0,0,0,0.0,0,0.0
99709,p-02792,66c52a77,x-00005,m-02730,2024-2025,2024-10-06,Home,t-00012,t-00064,LB,...,0,0,0,0,0,0,0,0.0,0,0.0
99710,p-02933,77d6fd4d,x-00005,m-02730,2024-2025,2024-10-06,Away,t-00064,t-00012,GK,...,0,0,0,0,0,57,7,16.8,3,0.0
99711,p-03102,74618572,x-00005,m-02730,2024-2025,2024-10-06,Home,t-00012,t-00064,LW,...,0,0,0,0,0,0,0,0.0,0,0.0


In [13]:
# List of stats columns to process
stats_columns = [
	"goals",
	"assists",
	"pens_made",
	"pens_att",
	"shots",
	"shots_on_target",
	"cards_yellow",
	"cards_red",
	"blocks",
	"xg",
	"npxg",
	"passes_total_distance",
	"passes_progressive_distance",
	"passes_completed_short",
	"passes_short",
	"passes_completed_medium",
	"passes_medium",
	"passes_completed_long",
	"passes_long",
	"xg_assist",
	"pass_xa",
	"assisted_shots",
	"passes_into_final_third",
	"passes_into_penalty_area",
	"crosses_into_penalty_area",
	"progressive_passes",
	"tackles",
	"tackles_won",
	"tackles_def_3rd",
	"tackles_mid_3rd",
	"tackles_att_3rd",
	"challenge_tackles",
	"challenges",
	"blocked_shots",
	"blocked_passes",
	"interceptions",
	"clearances",
	"errors",
	"touches",
	"touches_def_pen_area",
	"touches_def_3rd",
	"touches_mid_3rd",
	"touches_att_3rd",
	"touches_att_pen_area",
	"touches_live_ball",
	"take_ons",
	"take_ons_won",
	"take_ons_tackled",
	"carries",
	"carries_distance",
	"carries_progressive_distance",
	"progressive_carries",
	"carries_into_final_third",
	"carries_into_penalty_area",
	"miscontrols",
	"dispossessed",
	"passes_received",
	"progressive_passes_received",
	"sca",
	"sca_passes_live",
	"sca_passes_dead",
	"sca_take_ons",
	"sca_shots",
	"sca_fouled",
	"sca_defense",
	"gca",
	"gca_passes_live",
	"gca_passes_dead",
	"gca_take_ons",
	"gca_shots",
	"gca_fouled",
	"gca_defense",
	"gk_shots_on_target_against",
	"gk_goals_against",
	"gk_saves",
	"gk_clean_sheets",
	"gk_psxg",
	"gk_pens_att",
	"gk_pens_allowed",
	"gk_pens_saved",
	"gk_pens_missed",
	"gk_passed_completed_launched",
	"gk_passes_launched",
	"gk_passes",
	"gk_passes_throws",
	"gk_passes_length_avg",
	"gk_goal_kicks",
	"gk_goal_kicks_length_avg",
]

In [14]:
# Initialize player ratings, RD, and volatility
players = season_data['player_id'].unique()
initial_rating = 3000  # Initial rating
initial_RD = 200       # Initial rating deviation (high uncertainty)
initial_volatility = 0.04  # Initial volatility
tau = 0.3  # Volatility parameter (controls fluctuation)

# Create a nested dictionary for ratings
ratings = {
	stat: {
		player: {
			'rating': initial_rating,
            'RD': initial_RD,
            'vol': initial_volatility
		} for player in players
	} for stat in stats_columns
}

In [15]:
# Glicko-2 constants
q = np.log(10) / 400  # q = ln(10)/400 ≈ 0.0057565

def g(phi):
    """Calculates the g(phi) factor."""
    return 1 / np.sqrt(1 + (3 * phi ** 2) / (np.pi ** 2))

def E(mu, mu_j, phi_j):
    """Calculates the expected score."""
    return 1 / (1 + np.exp(-g(phi_j) * (mu - mu_j)))

def f(x, delta, phi, v, sigma, tau):
    """Function to solve for new volatility (sigma)."""
    a = np.log(sigma ** 2)
    ex = np.exp(x)
    numerator = ex * (delta ** 2 - phi ** 2 - v - ex)
    denominator = 2 * (phi ** 2 + v + ex) ** 2
    return (numerator / denominator) - ((x - a) / (tau ** 2))

def position_weight(player_position, opponent_position):
    """
    Calculate weight based on positional interactions.

    Returns a weight between 0 and 1.
    """
    # Position interactions
    if len(player_position.split(',')) > 1:
        player_position = player_position.split(',')[0]
    if len(opponent_position.split(',')) > 1:
        opponent_position = opponent_position.split(',')[0]
        
    player_position = player_position[0] if player_position not in ['MF', 'DF',] else player_position
    if player_position == 'MF':
        player_position = 'M'
    if player_position == 'DF':
        player_position = 'B'
    opponent_position = opponent_position[0] if opponent_position not in ['MF', 'DF',] else opponent_position
    if opponent_position == 'MF':
        opponent_position = 'M'
    if opponent_position == 'DF':
       opponent_position = 'B'
    
    position_interactions = {
        'W': {'B': 1.0, 'M': 0.5, 'W': 0.1, 'K': 0.05},
        'M': {'B': 0.5, 'M': 1.0, 'W': 0.5, 'K': 0.01},
        'B': {'B': 0.1, 'M': 0.5, 'W': 1.0, 'K': 0.0},
        'K': {'B': 0.1, 'M': 0.5, 'W': 1.0, 'K': 0.0},
    }

    # Side interactions
    side_interactions = {
        'L': {'L': 1.0, 'C': 0.5, 'R': 0.1},
        'C': {'L': 0.5, 'C': 1.0, 'R': 0.5},
        'R': {'L': 0.1, 'C': 0.5, 'R': 1.0},
    }

    # Get position weight
    pos_weight = position_interactions.get(player_position, {}).get(opponent_position, 0.5)

    # # Get side weight
    # side_weight = side_interactions.get(player_side, {}).get(opponent_side, 0.5)
    side_weight = pos_weight

    # Combine weights (average of position and side weights)
    weight = (pos_weight + side_weight) / 2

    return weight

In [16]:
# Process matches in chronological order
match_ids = sorted(season_data['match_id'].unique())

for match_id in match_ids:
    # Extract season_data for the current match
    stats_to_date: pd.DataFrame = season_data[season_data["match_id"] < match_id]
    match_season_data = season_data[season_data['match_id'] == match_id]
    teams = match_season_data['team_id'].unique()
    if len(teams) != 2:
        continue  # Skip if not exactly two teams

    # Identify teams and their players
    team1, team2 = teams
    team1_players = match_season_data[match_season_data['team_id'] == team1]
    team2_players = match_season_data[match_season_data['team_id'] == team2]
    
    ratings_history = []

    # Process each player in the match
    for idx, row in match_season_data.iterrows():
        date = row["date"]
        player = row['player_id']
        team = row['team_id']
        player_position = row['position']
        opponent_team = team2_players if team == team1 else team1_players
        
        player_ratings = {
            "date": date,
            "match_id": match_id,
            "player": player,
            "position": player_position,
		}

        for stat in stats_columns:
            # Skip if the stat value is NaN
            if pd.isna(row[stat]):
                continue

            # Retrieve player's current rating, RD, and volatility
            rating_dict = ratings[stat][player]
            rating_i = rating_dict['rating']
            RD_i = rating_dict['RD']
            sigma_i = rating_dict['vol']

            # Convert to Glicko-2 scale
            mu_i = (rating_i - 1500) / 173.7178
            phi_i = RD_i / 173.7178
            sigma_i = sigma_i  # Volatility remains unchanged

            # Initialize sums for variance and delta calculations
            sum_g_E = 0
            sum_g_s_E = 0
            
			# Get max stat value
            max_stat_value = stats_to_date[stat].max()

            # Iterate over opponents
            for opp_idx, opp_row in opponent_team.iterrows():
                opponent = opp_row['player_id']
                opponent_position = opp_row['position']
                opponent_rating = ratings[stat][opponent]['rating']
                opponent_RD = ratings[stat][opponent]['RD']
                mu_j = (opponent_rating - 1500) / 173.7178
                phi_j = opponent_RD / 173.7178

                # # Calculate weight based on positional interactions
                weight = position_weight(player_position, opponent_position)

                # Compute g(phi_j) and E(mu_i, mu_j, phi_j)
                g_phi_j = g(phi_j)
                E_ij = E(mu_i, mu_j, phi_j)

                # Actual performance: Normalize the stat value between 0 and 1
                if max_stat_value == 0:
                    continue  # Avoid division by zero
                actual_performance = row[stat] / max_stat_value

                # Update sums with weights
                sum_g_E += (g_phi_j ** 2) * E_ij * (1 - E_ij) * weight
                sum_g_s_E +=  g_phi_j * (actual_performance - E_ij) * weight

            if sum_g_E == 0:
                continue  # Avoid division by zero

            # Compute variance v
            v = 1 / sum_g_E

            # Compute delta (Δ)
            delta = v * sum_g_s_E

            # Update volatility (σ') using iterative method
            a = np.log(sigma_i ** 2)
            A = a
            B = None
            if delta ** 2 > phi_i ** 2 + v:
                B = np.log(delta ** 2 - phi_i ** 2 - v)
            else:
                k = 1
                while f(a - k * tau, delta, phi_i, v, sigma_i, tau) < 0:
                    k += 1
                B = a - k * tau

            # Newton-Raphson iteration to find σ'
            epsilon = 1e-6
            fA = f(A, delta, phi_i, v, sigma_i, tau)
            fB = f(B, delta, phi_i, v, sigma_i, tau)
            while abs(B - A) > epsilon:
                C = A + (A - B) * fA / (fB - fA)
                fC = f(C, delta, phi_i, v, sigma_i, tau)
                if fC * fB < 0:
                    A = B
                    fA = fB
                else:
                    fA /= 2
                B = C
                fB = fC

            sigma_prime = np.exp(A / 2)

            # Update phi* (intermediate RD)
            phi_star = np.sqrt(phi_i ** 2 + sigma_prime ** 2)

            # Update phi' (new RD)
            phi_prime = 1 / np.sqrt( (1 / phi_star ** 2) + (1 / v) )

            # Update mu' (new rating)
            mu_prime = mu_i + phi_prime ** 2 * sum_g_s_E

            # Convert back to original scale
            rating_prime = mu_prime * 173.7178 + 1500
            RD_prime = phi_prime * 173.7178

            # Update player's rating, RD, and volatility
            ratings[stat][player]['rating'] = rating_prime
            ratings[stat][player]['RD'] = RD_prime
            ratings[stat][player]['vol'] = sigma_prime
            
            player_ratings[stat] = [rating_prime, RD_prime, sigma_prime]
        ratings_history.append(player_ratings)


# Display the final ratings
for stat in stats_columns:
    print(f"\nFinal ratings for stat '{stat}':")
    for player, info in ratings[stat].items():
        print(f"  {player}: Rating={info['rating']:.2f}, RD={info['RD']:.2f}, Volatility={info['vol']:.5f}")


Final ratings for stat 'goals':
  p-02911: Rating=nan, RD=nan, Volatility=0.04000
  p-02853: Rating=nan, RD=nan, Volatility=0.04000
  p-02806: Rating=nan, RD=nan, Volatility=0.04000
  p-02829: Rating=nan, RD=nan, Volatility=0.04000
  p-02897: Rating=nan, RD=nan, Volatility=0.04000
  p-02919: Rating=nan, RD=nan, Volatility=0.04000
  p-02866: Rating=nan, RD=nan, Volatility=0.04000
  p-02871: Rating=nan, RD=nan, Volatility=0.04000
  p-02770: Rating=nan, RD=nan, Volatility=0.04000
  p-02717: Rating=nan, RD=nan, Volatility=0.04000
  p-02728: Rating=nan, RD=nan, Volatility=0.04000
  p-02775: Rating=nan, RD=nan, Volatility=0.04000
  p-02782: Rating=nan, RD=nan, Volatility=0.04000
  p-02750: Rating=nan, RD=nan, Volatility=0.04000
  p-02755: Rating=nan, RD=nan, Volatility=0.04000
  p-02760: Rating=nan, RD=nan, Volatility=0.04000
  p-02767: Rating=nan, RD=nan, Volatility=0.04000
  p-02768: Rating=nan, RD=nan, Volatility=0.04000
  p-03083: Rating=nan, RD=nan, Volatility=0.04000
  p-03088: Rating